# Pedestrian signal detector: first experiment

Trains a small detector for US pedestrian signals (walk, don't walk, countdown) and checks how often it says **walk** when it shouldn't.

## Before you run
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Add your Roboflow key as a secret:** open the Secrets panel (key icon in the left sidebar, labelled "Secrets"), choose **Add new secret**, name it `ROBOFLOW_API_KEY`, paste the private key as the value, and turn on **Notebook access**. The key never appears in the notebook.
3. **Runtime → Run all.** Training takes roughly an hour. Keep the tab open; free Colab disconnects idle sessions.
4. At the end the browser downloads `signal_detector_results.zip`. Put it in the audio_map scratchpad or anywhere and tell Claude where it is.

Every cell prints a plain-text summary, so you can follow progress with a screen reader without looking at plots.

## Data (all CC BY 4.0, from Roboflow Universe)
- brimagelle-eyewear/brimagelle-test1 v3: US street-level iPhone videos
- bibek-pjlvk/pedestrian-light-classifier v4: US city crossings, many at night
- garage-dkgp8/pedestrian-signals-gyb3d v1: US walk / don't walk close-ups
- yolov7-tldetection/lisa-traffic-light-detection-wusg9 v11: LISA car traffic lights (San Diego), used only as a test of false "walk"

Note: the Ultralytics library is AGPL-3.0. Fine for this experiment; the licence question has to be settled before any model ships in the app.

In [ ]:
!pip -q install ultralytics
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE. Choose Runtime, Change runtime type, T4 GPU, then run again.')
from google.colab import userdata
KEY = userdata.get('ROBOFLOW_API_KEY')
print('Roboflow key found' if KEY else 'Roboflow key missing: add the ROBOFLOW_API_KEY secret')

In [ ]:
# Download each dataset in YOLO format from the Roboflow API.
import json, time, zipfile, urllib.request, pathlib
RAW = pathlib.Path('/content/raw'); RAW.mkdir(exist_ok=True)
SOURCES = {
    'brim':   'brimagelle-eyewear/brimagelle-test1/3',
    'bibek':  'bibek-pjlvk/pedestrian-light-classifier/4',
    'garage': 'garage-dkgp8/pedestrian-signals-gyb3d/1',
    'lisa':   'yolov7-tldetection/lisa-traffic-light-detection-wusg9/11',
}
def fetch(name, ref):
    out = RAW / name
    if (out / 'data.yaml').exists():
        return out
    for _ in range(60):
        with urllib.request.urlopen(f'https://api.roboflow.com/{ref}/yolov8?api_key={KEY}') as r:
            link = json.load(r).get('export', {}).get('link')
        if link:
            break
        time.sleep(10)  # Roboflow is still preparing this export
    else:
        raise RuntimeError(f'{name}: export was not ready after 10 minutes')
    z = RAW / f'{name}.zip'
    urllib.request.urlretrieve(link, z)
    zipfile.ZipFile(z).extractall(out)
    z.unlink()
    return out
for name, ref in SOURCES.items():
    fetch(name, ref)
    print('downloaded', name)

In [ ]:
# Merge the three US sets into one, re-split by source video / photo so near-identical
# frames can't sit in both train and test, and keep one copy of each photo in valid/test.
import re, hashlib, shutil, collections, yaml
CLASSES = ['walk', 'dont_walk', 'countdown']
MAPS = {
    'brim':   {'crosswalklightgo': 0, 'crosswalklightstop': 1, 'crosswalklightcountdown': 2},
    'bibek':  {'white pedestrian light': 0, 'red pedestrian light': 1},
    'garage': {'Walk': 0, 'Don-t Walk': 1},
}
OUT = pathlib.Path('/content/ped')
shutil.rmtree(OUT, ignore_errors=True)

def names_of(src):
    n = yaml.safe_load(open(RAW / src / 'data.yaml'))['names']
    return n if isinstance(n, list) else [n[k] for k in sorted(n)]

def source_name(stem):  # Roboflow names files <original>_<ext>.rf.<hash>
    return stem.split('.rf.')[0]

def group_of(src, stem):
    n = source_name(stem)
    m = re.search(r'IMG_\d+', n) if src == 'brim' else None
    return m.group(0) if m else n

def assign_splits(src, entries):
    # Whole groups (videos / photos) in a fixed shuffled order: ~20% of distinct photos to test, ~10% to valid.
    size = collections.Counter()
    for g, n in {(group_of(src, i.stem), source_name(i.stem)) for _, i in entries}:
        size[g] += 1
    order = sorted(size, key=lambda g: hashlib.md5(f'{src}/{g}'.encode()).hexdigest())
    total, done, out = sum(size.values()), 0, {}
    for g in order:
        out[g] = 'test' if done < 0.2 * total else 'valid' if done < 0.3 * total else 'train'
        done += size[g]
    return out

lists = collections.defaultdict(list)
stats = collections.defaultdict(collections.Counter)
for src, cmap in MAPS.items():
    names = names_of(src)
    unknown = set(cmap) - set(names)
    assert not unknown, f'{src}: class names changed upstream: {unknown}'
    remap = {i: cmap[n] for i, n in enumerate(names) if n in cmap}
    entries = []
    for orig in ('test', 'valid', 'train'):  # unaugmented copies first
        for img in sorted((RAW / src / orig / 'images').glob('*')):
            entries.append((orig, img))
    split_of = assign_splits(src, entries)
    kept_eval = set()
    for orig, img in entries:
        grp = group_of(src, img.stem)
        new = split_of[grp]
        if new != 'train':
            key = source_name(img.stem)
            if key in kept_eval:
                continue
            kept_eval.add(key)
        lab = RAW / src / orig / 'labels' / (img.stem + '.txt')
        lines = []
        for line in (lab.read_text().splitlines() if lab.exists() else []):
            p = line.split()
            if p and int(p[0]) in remap:
                lines.append(' '.join([str(remap[int(p[0])])] + p[1:5]))
                stats[(src, new)][CLASSES[remap[int(p[0])]]] += 1
        dst = OUT / 'images' / new / f'{src}__{img.name}'
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(img, dst)
        ldst = OUT / 'labels' / new / f'{src}__{img.stem}.txt'
        ldst.parent.mkdir(parents=True, exist_ok=True)
        ldst.write_text('\n'.join(lines))
        stats[(src, new)]['images'] += 1
        lists[new].append(str(dst)); lists[f'{new}_{src}'].append(str(dst))

(OUT / 'lists').mkdir(exist_ok=True)
for k, v in lists.items():
    (OUT / 'lists' / f'{k}.txt').write_text('\n'.join(v))
def data_yaml(test_list='test'):
    p = OUT / f'data_{test_list}.yaml'
    p.write_text(yaml.safe_dump({'path': str(OUT), 'train': 'lists/train.txt', 'val': 'lists/valid.txt',
                                 'test': f'lists/{test_list}.txt', 'names': dict(enumerate(CLASSES))}))
    return str(p)
DATA = data_yaml()
for (src, sp), c in sorted(stats.items()):
    print(f'{src:7} {sp:6} images {c["images"]:5}  walk {c["walk"]:5}  dont_walk {c["dont_walk"]:5}  countdown {c["countdown"]:4}')

In [ ]:
# Train. Hue augmentation is off: colour is part of what makes a walk signal a walk signal.
from ultralytics import YOLO
IMGSZ = 960  # signals in street photos are only a few pixels wide; a larger input helps
model = YOLO('yolo11n.pt')
model.train(data=DATA, epochs=40, patience=10, imgsz=IMGSZ, batch=16, hsv_h=0.0,
            project='/content/runs', name='ped', exist_ok=True, verbose=False)
print('training finished')

In [ ]:
# Standard scores on the held-out test set, overall and per source.
import gc, torch
globals().pop('model', None)  # free the training run's GPU memory
gc.collect(); torch.cuda.empty_cache()
best = YOLO('/content/runs/ped/weights/best.pt')
report = {'per_source': {}}
for src in ['all', 'brim', 'bibek', 'garage']:
    lst = 'test' if src == 'all' else f'test_{src}'
    if not (OUT / 'lists' / f'{lst}.txt').exists():
        continue
    m = best.val(data=data_yaml(lst), split='test', imgsz=IMGSZ, verbose=False,
                 plots=(src == 'all'), project='/content/runs', name=f'test_{src}', exist_ok=True)
    per = {CLASSES[int(i)]: round(float(a), 3) for i, a in zip(m.box.ap_class_index, m.box.ap50)}
    report['per_source'][src] = {'mAP50': round(float(m.box.map50), 3), 'per_class_mAP50': per}
    print(f'{src:7} mAP50 {m.box.map50:.3f}  ', per)

In [ ]:
# Safety check: how often does it say WALK when it shouldn't?
import numpy as np
from PIL import Image
ERR = pathlib.Path('/content/out/errors'); shutil.rmtree(ERR, ignore_errors=True); ERR.mkdir(parents=True)

def gt_boxes(img_path, label_path):
    w, h = Image.open(img_path).size
    out = []
    if pathlib.Path(label_path).exists():
        for line in pathlib.Path(label_path).read_text().splitlines():
            p = line.split()
            if len(p) >= 5:
                c, cx, cy, bw, bh = int(p[0]), *map(float, p[1:5])
                out.append((c, ((cx - bw/2)*w, (cy - bh/2)*h, (cx + bw/2)*w, (cy + bh/2)*h)))
    return out

def label_for(img_path):  # .../images/<split>/x.jpg -> .../labels/<split>/x.txt
    parts = list(pathlib.Path(img_path).parts)
    i = len(parts) - 1 - parts[::-1].index('images')
    parts[i] = 'labels'
    return pathlib.Path(*parts).with_suffix('.txt')

def overlaps(a, b):  # tiny boxes: count any overlap of at least 20% of the smaller box
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    small = min((a[2]-a[0])*(a[3]-a[1]), (b[2]-b[0])*(b[3]-b[1])) or 1
    return ix * iy / small >= 0.2

def preds(paths, chunk=16):
    # A whole list is predicted as one batch, which runs out of GPU memory; go 16 pictures at a time.
    for i in range(0, len(paths), chunk):
        for r in best.predict(paths[i:i + chunk], stream=True, conf=0.25, imgsz=IMGSZ, verbose=False):
            yield r, [(int(c), float(s), tuple(b)) for c, s, b in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist())]

# 1) held-out pedestrian test set
test_imgs = (OUT / 'lists' / 'test.txt').read_text().split()
rows = []
for r, ps in preds(test_imgs):
    lab = label_for(r.path)
    gts = gt_boxes(r.path, lab)
    for c, s, b in ps:
        if c != 0:
            continue
        kind = ('correct' if any(g == 0 and overlaps(b, gb) for g, gb in gts) else
                'on_dont_walk' if any(g in (1, 2) and overlaps(b, gb) for g, gb in gts) else 'elsewhere')
        rows.append((s, kind, r))
    n_walk_gt = sum(1 for g, _ in gts if g == 0)
    rows.append((None, 'gt_walk', n_walk_gt))
total_gt_walk = sum(x[2] for x in rows if x[1] == 'gt_walk')
walk_preds = [x for x in rows if x[1] != 'gt_walk']
report['walk_safety_test'] = {}
for t in (0.25, 0.5, 0.7, 0.85):
    sel = [x for x in walk_preds if x[0] >= t]
    c = collections.Counter(x[1] for x in sel)
    report['walk_safety_test'][t] = dict(c, walk_signals_in_test=total_gt_walk)
    print(f'conf>={t}: walk said {len(sel)} times; correct {c["correct"]}, ON A DONT-WALK/COUNTDOWN {c["on_dont_walk"]}, '
          f'on nothing labelled {c["elsewhere"]} (of {total_gt_walk} real walk signals)')
for i, (s, kind, r) in enumerate(sorted([x for x in walk_preds if x[1] != 'correct'], key=lambda x: -x[0])[:25]):
    Image.fromarray(r.plot()[..., ::-1]).save(ERR / f'test_{kind}_{s:.2f}_{i}.jpg')

# 2) LISA car traffic lights: any walk on a car light is a confirmed mistake
lisa_names = names_of('lisa')
lisa_imgs = [str(p) for sp in ('test', 'valid') for p in sorted((RAW / 'lisa' / sp / 'images').glob('*'))]
lc = collections.Counter(); examples = []
for r, ps in preds(lisa_imgs):
    lab = label_for(r.path)
    lights = [(lisa_names[g], gb) for g, gb in gt_boxes(r.path, lab)]
    for c, s, b in ps:
        on = [n for n, gb in lights if overlaps(b, gb)]
        key = f'{CLASSES[c]}_on_car_light' if on else f'{CLASSES[c]}_elsewhere'
        for t in (0.25, 0.5, 0.7, 0.85):
            if s >= t:
                lc[(t, key)] += 1
        if c == 0 and s >= 0.5:
            examples.append((s, key, r))
report['lisa'] = {'images': len(lisa_imgs)}
for t in (0.25, 0.5, 0.7, 0.85):
    report['lisa'][t] = {k: v for (tt, k), v in lc.items() if tt == t}
    print(f'LISA {len(lisa_imgs)} car-view images, conf>={t}: walk on a car light {lc[(t, "walk_on_car_light")]}, '
          f'walk elsewhere {lc[(t, "walk_elsewhere")]} (may be real unlabelled pedestrian signals, check the pictures)')
for i, (s, key, r) in enumerate(sorted(examples, key=lambda x: -x[0])[:25]):
    Image.fromarray(r.plot()[..., ::-1]).save(ERR / f'lisa_{key}_{s:.2f}_{i}.jpg')

In [ ]:
# Export to ONNX (for onnxruntime-web in the app) and download everything as one zip.
onnx_path = best.export(format='onnx', imgsz=IMGSZ, opset=17, simplify=True)
O = pathlib.Path('/content/out')
for f in ['/content/runs/ped/weights/best.pt', onnx_path, '/content/runs/ped/results.csv', '/content/runs/ped/results.png',
          '/content/runs/test_all/confusion_matrix.png', '/content/runs/test_all/confusion_matrix_normalized.png']:
    if pathlib.Path(f).exists():
        shutil.copy(f, O)
(O / 'report.json').write_text(json.dumps(report, indent=1, default=str))
(O / 'split_stats.txt').write_text('\n'.join(f'{s} {sp} {dict(c)}' for (s, sp), c in sorted(stats.items())))
shutil.make_archive('/content/signal_detector_results', 'zip', O)
from google.colab import files
files.download('/content/signal_detector_results.zip')
print('done: signal_detector_results.zip is downloading')